In [ ]:
# ===== LOCAL WINDOWS BOOTSTRAP (added by adapt_notebooks.py) =====
# This cell replaces the Kaggle-specific path setup of "Stable Diffusion pipelines in Diffusers".
import os
from pathlib import Path

# LAB_DIR = the lab3/ folder that contains this notebook (notebooks/ is one level deeper).
LAB_DIR = Path.cwd()
# When the notebook is launched from lab3/notebooks/, .. is the lab root.
if LAB_DIR.name == "notebooks":
    LAB_DIR = LAB_DIR.parent

WORK_DIR = LAB_DIR / "work"
DATA_DIR = LAB_DIR / "data"
OUTPUTS_DIR = LAB_DIR / "outputs"
for d in (WORK_DIR, DATA_DIR, OUTPUTS_DIR):
    d.mkdir(parents=True, exist_ok=True)

os.environ["LAB3_WORK_DIR"] = str(WORK_DIR)
os.environ["LAB3_DATA_DIR"] = str(DATA_DIR)

# Keep Hugging Face caches inside the lab folder so we don't pollute %USERPROFILE%.
os.environ.setdefault("HF_HOME",            str(WORK_DIR / ".cache" / "huggingface"))
os.environ.setdefault("HF_HUB_CACHE",       str(WORK_DIR / ".cache" / "huggingface" / "hub"))
os.environ.setdefault("TRANSFORMERS_CACHE", str(WORK_DIR / ".cache" / "huggingface" / "transformers"))
os.environ.setdefault("DIFFUSERS_CACHE",    str(WORK_DIR / ".cache" / "huggingface" / "diffusers"))
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")

print("LAB_DIR    :", LAB_DIR)
print("WORK_DIR   :", WORK_DIR)
print("DATA_DIR   :", DATA_DIR)
print("OUTPUTS_DIR:", OUTPUTS_DIR)


# Лабораторна робота: Stable Diffusion pipelines у Diffusers

**Навчальний Kaggle-notebook**  
Адаптовано на основі Hugging Face Diffusion Course, Unit 3, lesson 2:  
https://huggingface.co/learn/diffusion-course/unit3/2

---

## Коротко про роботу

У цій лабораторній роботі ми вивчаємо, як працюють основні pipeline Stable Diffusion у бібліотеці **Diffusers**:

1. **Text-to-Image** — генерація зображення з текстового prompt.
2. **Pipeline components** — VAE, tokenizer, text encoder, UNet, scheduler.
3. **DIY sampling loop** — ручна реалізація базового diffusion sampling.
4. **Img2Img** — зміна початкового зображення за prompt.
5. **Inpainting** — заміна частини зображення за маскою.
6. **Depth2Img** — генерація з урахуванням depth map.
7. **Порівняння результатів і підготовка звіту.**

Notebook зроблено навчальним: кожен блок має пояснення, завдання для студента, контрольні питання та місце для аналізу.

# 1. Мета лабораторної

Після виконання роботи студент має вміти:

- запускати Stable Diffusion pipeline у Kaggle;
- пояснювати роль `prompt`, `negative_prompt`, `guidance_scale`, `num_inference_steps`, `seed`;
- пояснювати роль основних компонентів pipeline:
  - `VAE`;
  - `tokenizer`;
  - `text_encoder`;
  - `UNet`;
  - `scheduler`;
- показати, як зображення стискається у latent-простір;
- пояснити, що означає scaling factor VAE;
- реалізувати спрощений sampling loop вручну;
- запустити `Img2Img`, `Inpainting`, `Depth2Img`;
- порівняти результати різних pipeline;
- оформити короткий технічний звіт.

---

## Очікуваний результат

У кінці роботи студент має здати:

1. виконаний notebook;
2. generated images;
3. таблицю параметрів;
4. порівняння результатів;
5. відповіді на контрольні питання;
6. короткий висновок про обмеження та етичні ризики.

# 2. Правила безпеки й дозволені типи даних

Stable Diffusion може створювати реалістичні зображення, тому в лабораторній роботі діють обмеження.

## Дозволено

- навчальні prompts без персональних даних;
- власні або відкриті зображення без людей;
- пейзажі, предмети, тварини, іграшки, рослини;
- синтетичні demo images;
- безпечне редагування зображень.

## Заборонено

- генерувати або редагувати реальних людей без згоди;
- створювати deepfake або impersonation;
- використовувати чужі приватні фото;
- створювати NSFW або шкідливий контент;
- використовувати copyrighted characters як об’єкт лабораторної;
- публікувати generated images без перевірки ліцензії та безпеки.

## Для цієї роботи

Ми використовуємо demo-зображення для Img2Img/Inpainting або створюємо простий fallback image автоматично.

# 3. Налаштування Kaggle

Перед запуском перевірте налаштування notebook:

```text
Settings → Accelerator → GPU
Settings → Internet → On
```

Якщо GPU недоступний, більшість cells працюватиме дуже повільно.  
Для навчальної роботи рекомендовано Kaggle GPU T4/P100 або аналогічний GPU.
Для цієї лабораторної рекомендований GPU: T4. Допустимий GPU: P100.

P100 має 16 GB HBM2 і високу пропускну здатність пам’яті, але поточна PyTorch-збірка Kaggle може не підтримувати P100, оскільки P100 має compute capability sm_60.
Якщо з’являється попередження про несумісність P100 із PyTorch, потрібно обрати T4.
T4 x2 не дає автоматичного прискорення, якщо notebook не налаштований на multi-GPU (налаштування не входить в матеріали даної роботи).

In [ ]:
# Перевірка GPU та середовища
import os
import sys
import platform
import torch

print("Python:", sys.version)
print("Platform:", platform.platform())
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    total_mem = torch.cuda.get_device_properties(0).total_memory / 1024**3
    print(f"GPU memory: {total_mem:.2f} GB")
else:
    print("GPU не знайдено. У Kaggle перевірте: Settings → Accelerator → GPU")

# 4. Встановлення залежностей

Ця cell встановлює бібліотеки:

- `diffusers`;
- `transformers`;
- `accelerate`;
- `safetensors`;
- `ftfy`;
- `huggingface_hub`;
- `matplotlib`;
- `pillow`.

Якщо cell виконується довго, це нормально: Kaggle встановлює або оновлює пакети.

In [ ]:
# (bash version-info cell stripped for local Windows run)


In [ ]:
# Installs dependencies. On Windows you can instead use the project requirements.txt.
%pip install diffusers transformers accelerate safetensors ftfy huggingface_hub pillow matplotlib requests tqdm xformers


In [ ]:
# Installs dependencies. On Windows you can instead use the project requirements.txt.
%pip install xformers


In [ ]:
# Перевірка встановлених версій
import importlib

packages = [
    "diffusers",
    "transformers",
    "accelerate",
    "huggingface_hub",
    "safetensors",
    "PIL",
    "matplotlib",
    "torch",
    "xformers"
]

for package in packages:
    try:
        module = importlib.import_module(package)
        version = getattr(module, "__version__", "unknown")
        print(f"{package}: {version}")
    except Exception as e:
        print(f"{package}: not available ({e})")

# 5. Авторизація Hugging Face

Деякі моделі можуть вимагати Hugging Face token або прийняття ліцензії на сторінці моделі.

## Рекомендовано для Kaggle

1. У Kaggle відкрийте **Add-ons → Secrets**.
2. Додайте secret з назвою:

```text
HF_TOKEN
```

3. Створіть власний Hugging Face token з правом `read` та вставте його у secret HF_TOKEN.
4. Увімкніть доступ notebook до secret.

Якщо модель завантажується без token, ця cell просто продовжить роботу.

In [ ]:
import os

# Local Windows fallback for Kaggle Secrets.
# Provide your HF token via environment variable HF_TOKEN before launching jupyter
# (e.g. in PowerShell: $env:HF_TOKEN="hf_xxx"; jupyter lab).
hf_token = os.environ.get("HF_TOKEN")
if hf_token:
    print("HF_TOKEN found in environment.")
else:
    print("HF_TOKEN not set. Public models still work; gated/private ones won't.")

from huggingface_hub import whoami, login
if hf_token:
    try:
        login(token=hf_token, add_to_git_credential=False)
        user = whoami(token=hf_token)
        print("Logged in as:", user.get("name", "unknown"))
    except Exception as e:
        print("HF login failed:", repr(e))
else:
    print("Continuing without Hugging Face authentication.")


# 6. Імпорти та базові налаштування

У цьому notebook використовується модель:

```text
stabilityai/stable-diffusion-2-1-base
```

Вона відповідає оригінальній логіці Hugging Face Unit 3/2.

Якщо у вас виникне проблема з доступом до моделі, викладач може замінити `MODEL_ID` на іншу доступну text-to-image модель, сумісну з Diffusers.

In [ ]:
#перевірка доступності моделей

from huggingface_hub import model_info

for repo_id in [
    "stable-diffusion-v1-5/stable-diffusion-v1-5",
    "stable-diffusion-v1-5/stable-diffusion-inpainting",
    "sd2-community/stable-diffusion-2-depth",
]:
    try:
        info = model_info(repo_id)
        print("OK:", repo_id)
        print("  private:", info.private)
        print("  gated:", info.gated)
    except Exception as e:
        print("ERROR:", repo_id)
        print(type(e).__name__, e)

In [ ]:
import gc
import math
import random
from pathlib import Path
from io import BytesIO

import numpy as np
import requests
from PIL import Image, ImageDraw
import matplotlib.pyplot as plt
import torch

from diffusers import (
    StableDiffusionPipeline,
    StableDiffusionImg2ImgPipeline,
    StableDiffusionInpaintPipeline,
    StableDiffusionDepth2ImgPipeline,
    LMSDiscreteScheduler,
)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DTYPE = torch.float16 if DEVICE == "cuda" else torch.float32

MODEL_ID = "stable-diffusion-v1-5/stable-diffusion-v1-5"
IMAGE_HEIGHT = 512
IMAGE_WIDTH = 512

# Для економії часу можна вимикати важкі секції
RUN_TEXT2IMG = True
RUN_DIY_SAMPLING = True
RUN_IMG2IMG = True
RUN_INPAINTING = True

# Depth2Img завантажує окрему важку модель. Для слабкого GPU можна залишити False.
RUN_DEPTH2IMG = True

OUTPUT_DIR = Path(os.environ["LAB3_WORK_DIR"]) / "stable_diffusion_lab_outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("DEVICE:", DEVICE)
print("DTYPE:", DTYPE)
print("MODEL_ID:", MODEL_ID)
print("OUTPUT_DIR:", OUTPUT_DIR)

# 7. Допоміжні функції

Тут зібрано функції для:

- відображення grid-зображень;
- збереження результатів;
- декодування latent tensor у PIL image;
- завантаження demo image з fallback-варіантом.

In [ ]:
def show_images(images, titles=None, cols=3, figsize=(15, 5)):
    """Показати список PIL images у вигляді grid."""
    if not isinstance(images, (list, tuple)):
        images = [images]
    n = len(images)
    cols = min(cols, n)
    rows = math.ceil(n / cols)
    plt.figure(figsize=(figsize[0], figsize[1] * rows))
    for i, img in enumerate(images):
        plt.subplot(rows, cols, i + 1)
        plt.imshow(img)
        if titles:
            plt.title(titles[i])
        plt.axis("off")
    plt.tight_layout()
    plt.show()


def save_images(images, prefix):
    """Зберегти список PIL images у OUTPUT_DIR."""
    if not isinstance(images, (list, tuple)):
        images = [images]
    paths = []
    for i, img in enumerate(images):
        path = OUTPUT_DIR / f"{prefix}_{i:02d}.png"
        img.save(path)
        paths.append(path)
    print("Saved:")
    for p in paths:
        print(" ", p)
    return paths


def tensor_to_pil(image_tensor):
    """
    Перетворити tensor у формат PIL image.
    Очікується tensor shape [B, 3, H, W] у діапазоні приблизно [-1, 1].
    """
    image = (image_tensor / 2 + 0.5).clamp(0, 1)
    image = image.detach().cpu().permute(0, 2, 3, 1).numpy()
    image = (image * 255).round().astype("uint8")
    return [Image.fromarray(img) for img in image]


def create_fallback_image(size=(512, 512)):
    """
    Створює просте synthetic demo image, якщо завантаження з інтернету недоступне.
    """
    img = Image.new("RGB", size, (210, 225, 240))
    draw = ImageDraw.Draw(img)

    # Небо/земля
    draw.rectangle([0, 0, size[0], int(size[1] * 0.58)], fill=(190, 220, 245))
    draw.rectangle([0, int(size[1] * 0.58), size[0], size[1]], fill=(120, 170, 120))

    # Лавка
    draw.rectangle([120, 310, 390, 330], fill=(120, 80, 45))
    draw.rectangle([130, 335, 380, 350], fill=(120, 80, 45))
    draw.rectangle([150, 350, 165, 430], fill=(80, 55, 35))
    draw.rectangle([345, 350, 360, 430], fill=(80, 55, 35))

    # Людина-схема
    draw.ellipse([230, 170, 285, 225], fill=(235, 190, 150), outline=(70, 60, 50), width=3)
    draw.rectangle([240, 225, 275, 310], fill=(80, 110, 180))
    draw.line([240, 310, 210, 380], fill=(40, 50, 80), width=8)
    draw.line([275, 310, 315, 380], fill=(40, 50, 80), width=8)
    draw.line([240, 245, 200, 285], fill=(80, 110, 180), width=7)
    draw.line([275, 245, 320, 285], fill=(80, 110, 180), width=7)

    # Просте дерево
    draw.rectangle([45, 210, 75, 420], fill=(95, 60, 35))
    draw.ellipse([0, 90, 130, 245], fill=(55, 130, 70))

    return img


def create_fallback_mask(size=(512, 512)):
    """
    Маска для inpainting: біла область буде замінюватися, чорна лишиться.
    """
    mask = Image.new("RGB", size, (0, 0, 0))
    draw = ImageDraw.Draw(mask)
    draw.rectangle([215, 155, 305, 325], fill=(255, 255, 255))
    return mask


def download_image_or_fallback(url, fallback_fn, size=(512, 512), timeout=20):
    try:
        response = requests.get(url, timeout=timeout)
        response.raise_for_status()
        img = Image.open(BytesIO(response.content)).convert("RGB")
        return img.resize(size)
    except Exception as e:
        print("Не вдалося завантажити image з URL. Використовуємо fallback.")
        print("Причина:", repr(e))
        return fallback_fn(size)


def clear_gpu():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

# 8. Demo image і mask для Img2Img/Inpainting

Оригінальний notebook Hugging Face використовує demo image і mask з GitHub.  
Якщо Kaggle Internet вимкнено або URL недоступний, створиться просте synthetic image.

In [ ]:
img_url = "https://raw.githubusercontent.com/CompVis/latent-diffusion/main/data/inpainting_examples/overture-creations-5sI6fQgYIuo.png"
mask_url = "https://raw.githubusercontent.com/CompVis/latent-diffusion/main/data/inpainting_examples/overture-creations-5sI6fQgYIuo_mask.png"

init_image = download_image_or_fallback(
    img_url,
    fallback_fn=create_fallback_image,
    size=(IMAGE_WIDTH, IMAGE_HEIGHT)
)

mask_image = download_image_or_fallback(
    mask_url,
    fallback_fn=create_fallback_mask,
    size=(IMAGE_WIDTH, IMAGE_HEIGHT)
)

show_images([init_image, mask_image], titles=["init_image", "mask_image"], cols=2, figsize=(10, 5))

init_image.save(OUTPUT_DIR / "demo_init_image.png")
mask_image.save(OUTPUT_DIR / "demo_mask_image.png")

# 9. Text-to-Image pipeline

Тепер завантажимо `StableDiffusionPipeline`.

## Параметри, які потрібно зрозуміти

| Параметр | Значення |
|---|---|
| `prompt` | що модель має створити |
| `negative_prompt` | чого модель має уникати |
| `guidance_scale` | наскільки сильно модель слідує prompt |
| `num_inference_steps` | кількість denoising-кроків |
| `generator` / `seed` | відтворюваність результату |
| `height`, `width` | розмір зображення, кратний 8 |

## Завдання студента

1. Запустіть базову генерацію.
2. Змініть prompt.
3. Змініть `guidance_scale`.
4. Порівняйте результати.

In [ ]:
clear_gpu()

pipe = StableDiffusionPipeline.from_pretrained(
    MODEL_ID,
    torch_dtype=DTYPE,
    use_safetensors=True,
)

pipe = pipe.to(DEVICE)

# Економія пам'яті
pipe.enable_attention_slicing()

try:
    pipe.enable_xformers_memory_efficient_attention()
    print("xFormers enabled.")
except Exception as e:
    print("xFormers не увімкнено. Це не критично:", repr(e))

print("Pipeline loaded:", type(pipe).__name__)
print("Scheduler:", type(pipe.scheduler).__name__)

In [ ]:
if RUN_TEXT2IMG:
    prompt = "Palette knife painting of an autumn cityscape"
    negative_prompt = "Oversaturated, blurry, low quality"

    generator = torch.Generator(device=DEVICE).manual_seed(42)

    output = pipe(
        prompt=prompt,
        negative_prompt=negative_prompt,
        height=512,
        width=512,
        guidance_scale=8.0,
        num_inference_steps=30,
        generator=generator,
    )

    text2img_image = output.images[0]
    show_images(text2img_image, titles=["Text-to-Image result"], cols=1)
    save_images(text2img_image, "text2img_basic")

# 10. Експеримент: вплив `guidance_scale`

`guidance_scale` керує тим, наскільки сильно результат має відповідати prompt.

- Низьке значення: модель має більше свободи, prompt може виконуватись слабше.
- Середнє значення: часто найкращий баланс.
- Дуже високе значення: prompt сильніший, але можливі артефакти.

## Завдання

Запустіть cell і опишіть, який результат найкращий.

In [ ]:
if RUN_TEXT2IMG:
    cfg_scales = [2.0, 8.0, 12.0]
    prompt = "A collie dog with a pink hat"
    negative_prompt = "blurry, low quality, distorted"

    cfg_images = []
    cfg_titles = []

    for cfg in cfg_scales:
        generator = torch.Generator(device=DEVICE).manual_seed(123)
        image = pipe(
            prompt=prompt,
            negative_prompt=negative_prompt,
            height=512,
            width=512,
            guidance_scale=cfg,
            num_inference_steps=30,
            generator=generator,
        ).images[0]
        cfg_images.append(image)
        cfg_titles.append(f"guidance_scale={cfg}")

    show_images(cfg_images, titles=cfg_titles, cols=3, figsize=(15, 5))
    save_images(cfg_images, "guidance_scale_experiment")

# 11. Компоненти Stable Diffusion pipeline

Stable Diffusion — це не одна модель, а система з кількох компонентів.

Основні компоненти:

| Компонент | Роль |
|---|---|
| `VAE` | стискає image у latent space і декодує latent назад в image |
| `tokenizer` | перетворює prompt у token IDs |
| `text_encoder` | перетворює tokens у embeddings |
| `UNet` | прогнозує шум, який треба прибрати |
| `scheduler` | керує denoising-кроками |
| `safety_checker` | може перевіряти небезпечний контент, якщо присутній |

## Завдання

Виведіть список компонентів і знайдіть їх у змінній `pipe`.

In [ ]:
def print_pipeline(pipe):
  print("Pipeline components:")
  for name, component in pipe.components.items():
    print(f"{name:20s} -> {type(component).__name__}")

print_pipeline(pipe)

# 12. VAE: image → latent → image

VAE стискає зображення з pixel space у latent space.

Для Stable Diffusion типове перетворення:

```text
[1, 3, 512, 512] → [1, 4, 64, 64]
```

Тобто VAE зменшує просторовий розмір у 8 разів.

## Важливий момент

Вхід до VAE має бути в діапазоні приблизно `[-1, 1]`, тому в коді часто є нормалізація:

```python
images = torch.rand(...) * 2 - 1
```

А latent масштабується через:

```python
pipe.vae.config.scaling_factor
```

Для багатьох Stable Diffusion моделей це близько `0.18215`.

In [ ]:
# Створюємо штучне "зображення" у форматі, який очікує VAE: [-1, 1]
images = torch.rand(
    1, 3, IMAGE_HEIGHT, IMAGE_WIDTH,
    device=DEVICE,
    dtype=DTYPE,
) * 2 - 1

scaling_factor = pipe.vae.config.scaling_factor

with torch.no_grad():
    raw_latents = pipe.vae.encode(images).latent_dist.mean
    latents = raw_latents * scaling_factor
    decoded_images = pipe.vae.decode(latents / scaling_factor).sample

print("images shape:", tuple(images.shape))
print("raw_latents shape:", tuple(raw_latents.shape))
print("scaled latents shape:", tuple(latents.shape))
print("decoded_images shape:", tuple(decoded_images.shape))
print("VAE scaling_factor:", scaling_factor)

In [ ]:
# Візуалізація random input і decoded output
original_pil = tensor_to_pil(images)[0]
decoded_pil = tensor_to_pil(decoded_images)[0]

show_images(
    [original_pil, decoded_pil],
    titles=["Random image before VAE", "Decoded after VAE"],
    cols=2,
    figsize=(10, 5)
)

## Контрольні питання до VAE

1. Чому Stable Diffusion працює в latent space, а не напряму з pixel space?
2. Що означає shape `[1, 4, 64, 64]`?
3. Навіщо потрібен `scaling_factor`?
4. Що буде, якщо забути поділити latents на `scaling_factor` перед VAE decoder?

# 13. Tokenizer і Text Encoder

Prompt не передається в UNet як текст. Він проходить два етапи:

```text
prompt → tokenizer → token IDs → text_encoder → text embeddings
```

Text embeddings потім використовуються як умова для UNet.

In [ ]:
prompt = "A painting of a flooble"

text_inputs = pipe.tokenizer(
    prompt,
    padding="max_length",
    max_length=pipe.tokenizer.model_max_length,
    truncation=True,
    return_tensors="pt",
)

input_ids = text_inputs.input_ids
tokens = [pipe.tokenizer.decode([token_id]) for token_id in input_ids[0][:20]]

print("Prompt:", prompt)
print("Tokenizer max length:", pipe.tokenizer.model_max_length)
print("input_ids shape:", tuple(input_ids.shape))
print("First 20 decoded tokens:")
print(tokens)

with torch.no_grad():
    text_embeddings = pipe.text_encoder(input_ids.to(DEVICE))[0]

print("text_embeddings shape:", tuple(text_embeddings.shape))

## Що важливо помітити

Слово, якого немає у словнику tokenizer, може бути розбите на частини.  
Це важливо для DreamBooth, Textual Inversion і LoRA: спеціальні token треба добирати уважно.

## Завдання

Змініть prompt на свій і подивіться, як tokenizer розбиває слова.

# 14. UNet: прогнозування шуму

UNet отримує:

```text
noisy latents + timestep + text embeddings
```

і прогнозує noise residual — шум, який треба прибрати на цьому кроці.

In [ ]:
# Підготуємо scheduler timesteps
pipe.scheduler.set_timesteps(30, device=DEVICE)
timestep = pipe.scheduler.timesteps[0]

# Створимо випадкові latents потрібної форми
latent_channels = pipe.unet.config.in_channels
latent_height = IMAGE_HEIGHT // pipe.vae_scale_factor
latent_width = IMAGE_WIDTH // pipe.vae_scale_factor

test_latents = torch.randn(
    1, latent_channels, latent_height, latent_width,
    device=DEVICE,
    dtype=DTYPE,
)

with torch.no_grad():
    noise_pred = pipe.unet(
        test_latents,
        timestep,
        encoder_hidden_states=text_embeddings.to(dtype=DTYPE),
    ).sample

print("test_latents shape:", tuple(test_latents.shape))
print("noise_pred shape:", tuple(noise_pred.shape))
print("timestep:", timestep)

## Контрольні питання до UNet

1. Чому форма `noise_pred` така сама, як форма `latents`?
2. Що таке `timestep`?
3. Чому UNet потрібні `text_embeddings`?

# 15. Scheduler

Scheduler керує процесом поступового прибирання шуму.

UNet прогнозує шум, а scheduler вирішує, як перейти:

```text
x_t → x_{t-1}
```

У Diffusers можна замінювати scheduler без зміни самої моделі.

In [ ]:
print("Current scheduler:", type(pipe.scheduler).__name__)
print("Scheduler config keys:")
print(list(pipe.scheduler.config.keys())[:20])

# Якщо scheduler має alphas_cumprod, побудуємо графік
if hasattr(pipe.scheduler, "alphas_cumprod"):
    alphas = pipe.scheduler.alphas_cumprod.detach().cpu().numpy()
    plt.figure(figsize=(8, 4))
    plt.plot(alphas)
    plt.title("Scheduler alphas_cumprod")
    plt.xlabel("training timestep")
    plt.ylabel("alpha cumulative product")
    plt.grid(True)
    plt.show()
else:
    print("Цей scheduler не має alphas_cumprod для прямої візуалізації.")

In [ ]:
# Приклад створення альтернативного scheduler
alternative_scheduler = LMSDiscreteScheduler.from_config(pipe.scheduler.config)
print("Alternative scheduler:", type(alternative_scheduler).__name__)

In [ ]:
from diffusers import DDIMScheduler

alternative_scheduler = DDIMScheduler.from_config(pipe.scheduler.config)
print("Alternative scheduler:", type(alternative_scheduler).__name__)

In [ ]:
from diffusers import DDIMScheduler, DDPMScheduler, EulerDiscreteScheduler, LMSDiscreteScheduler, PNDMScheduler

scheduler_examples = {
    "Default": pipe.scheduler,
    "DDIM": DDIMScheduler.from_config(pipe.scheduler.config),
    "DDPM": DDPMScheduler.from_config(pipe.scheduler.config),
    "Euler": EulerDiscreteScheduler.from_config(pipe.scheduler.config),
    "LMS": LMSDiscreteScheduler.from_config(pipe.scheduler.config),
    "PNDM": PNDMScheduler.from_config(pipe.scheduler.config),
}

for name, scheduler in scheduler_examples.items():
    print(f"{name}: {type(scheduler).__name__}")

# Не замінюємо scheduler глобально автоматично, щоб не ламати подальші cells.
# Якщо потрібно протестувати:
# pipe.scheduler = LMSDiscreteScheduler.from_config(pipe.scheduler.config)

# 16. Helper для prompt embeddings

У різних версіях Diffusers методи кодування prompt можуть трохи відрізнятися.  
Тому створимо helper, який повертає embeddings для classifier-free guidance.

In [ ]:
def get_prompt_embeds(pipe, prompt, negative_prompt="", num_images_per_prompt=1):
    """
    Повертає embeddings у форматі:
    [negative_prompt_embeds, prompt_embeds]
    для classifier-free guidance.
    """
    do_classifier_free_guidance = True

    if hasattr(pipe, "encode_prompt"):
        result = pipe.encode_prompt(
            prompt=prompt,
            device=DEVICE,
            num_images_per_prompt=num_images_per_prompt,
            do_classifier_free_guidance=do_classifier_free_guidance,
            negative_prompt=negative_prompt,
        )
        # У сучасних diffusers повертається tuple: (prompt_embeds, negative_prompt_embeds, ...)
        prompt_embeds = result[0]
        negative_prompt_embeds = result[1]
        text_embeddings = torch.cat([negative_prompt_embeds, prompt_embeds])
        return text_embeddings

    # Fallback для старіших версій
    return pipe._encode_prompt(
        prompt,
        DEVICE,
        num_images_per_prompt,
        do_classifier_free_guidance,
        negative_prompt,
    )

# 17. DIY Sampling Loop: ручна реалізація Text-to-Image

Тепер повторимо те, що робить `StableDiffusionPipeline`, але вручну.

## Алгоритм

```text
1. Закодувати prompt.
2. Створити випадкові latents.
3. Налаштувати scheduler timesteps.
4. Для кожного timestep:
   - продублювати latents для classifier-free guidance;
   - подати latents у UNet;
   - отримати noise prediction;
   - застосувати guidance_scale;
   - виконати scheduler.step().
5. Декодувати latents через VAE.
```

Цей блок важливий для розуміння внутрішньої логіки Diffusion.

In [ ]:
def decode_latents_to_pil(pipe, latents):
    """
    Декодує scaled latents Stable Diffusion у PIL images.
    """
    latents = latents / pipe.vae.config.scaling_factor

    with torch.no_grad():
        image = pipe.vae.decode(latents, return_dict=False)[0]

    image = (image / 2 + 0.5).clamp(0, 1)
    image = image.detach().cpu().permute(0, 2, 3, 1).numpy()
    image = (image * 255).round().astype("uint8")
    return [Image.fromarray(img) for img in image]


def diy_text2img(
    pipe,
    prompt,
    negative_prompt="",
    height=512,
    width=512,
    num_inference_steps=30,
    guidance_scale=8.0,
    seed=42,
):
    generator = torch.Generator(device=DEVICE).manual_seed(seed)

    text_embeddings = get_prompt_embeds(
        pipe,
        prompt=prompt,
        negative_prompt=negative_prompt,
        num_images_per_prompt=1,
    ).to(dtype=DTYPE)

    pipe.scheduler.set_timesteps(num_inference_steps, device=DEVICE)

    latent_shape = (
        1,
        pipe.unet.config.in_channels,
        height // pipe.vae_scale_factor,
        width // pipe.vae_scale_factor,
    )

    latents = torch.randn(
        latent_shape,
        generator=generator,
        device=DEVICE,
        dtype=DTYPE,
    )

    latents = latents * pipe.scheduler.init_noise_sigma

    for t in pipe.scheduler.timesteps:
        # 1) duplicate latents for unconditional + text-conditioned prediction
        latent_model_input = torch.cat([latents] * 2)

        # 2) scheduler-specific scaling
        latent_model_input = pipe.scheduler.scale_model_input(latent_model_input, t)

        # 3) predict noise residual
        with torch.no_grad():
            noise_pred = pipe.unet(
                latent_model_input,
                t,
                encoder_hidden_states=text_embeddings,
            ).sample

        # 4) classifier-free guidance
        noise_pred_uncond, noise_pred_text = noise_pred.chunk(2)
        noise_pred = noise_pred_uncond + guidance_scale * (
            noise_pred_text - noise_pred_uncond
        )

        # 5) scheduler step
        latents = pipe.scheduler.step(noise_pred, t, latents).prev_sample

    return decode_latents_to_pil(pipe, latents)[0]

In [ ]:
if RUN_DIY_SAMPLING:
    diy_prompt = "A watercolor painting of a small robot reading a book"
    diy_negative_prompt = "blurry, low quality, distorted"

    diy_image = diy_text2img(
        pipe=pipe,
        prompt=diy_prompt,
        negative_prompt=diy_negative_prompt,
        height=512,
        width=512,
        num_inference_steps=30,
        guidance_scale=8.0,
        seed=42,
    )

    show_images(diy_image, titles=["DIY Text-to-Image sampling"], cols=1)
    save_images(diy_image, "diy_text2img")

## Контрольні питання до DIY sampling

1. На якому етапі використовується prompt?
2. Навіщо latents дублюються через `torch.cat([latents] * 2)`?
3. Що означають `noise_pred_uncond` і `noise_pred_text`?
4. Що робить формула classifier-free guidance?
5. Чому після sampling треба декодувати latents через VAE?

# 18. Img2Img pipeline

`Img2Img` не починає з повністю випадкового шуму.  
Він бере початкове зображення, кодує його в latent space, додає шум, а потім виконує denoising за prompt.

## Основна формула

```text
input image
→ VAE encoder
→ image latents
→ add noise
→ denoising with prompt
→ VAE decoder
→ output image
```

## Головний параметр: `strength`

| `strength` | Що відбувається |
|---|---|
| 0.2–0.3 | слабкі зміни, зберігається input |
| 0.5–0.7 | помітні зміни, структура ще зберігається |
| 0.8–1.0 | сильні зміни, input майже зникає |

## Завдання

Запустіть Img2Img із різними `strength` і порівняйте результати.

## 18.1 DIY Img2Img loop: ручна логіка

У цьому блоці ми вручну повторюємо головну ідею Img2Img:

```text
input image
→ VAE encoder
→ image latents
→ додавання шуму залежно від strength
→ denoising loop
→ VAE decoder
→ output image
```

Цей код потрібен не для найкращої якості, а для розуміння механіки.

In [ ]:
def prepare_pil_image_for_vae(image, height=512, width=512):
    """
    PIL image → torch tensor у форматі [1, 3, H, W] і діапазоні [-1, 1].
    """
    image = image.convert("RGB").resize((width, height))
    image_np = np.array(image).astype(np.float32) / 255.0
    image_np = image_np.transpose(2, 0, 1)
    image_tensor = torch.from_numpy(image_np).unsqueeze(0)
    image_tensor = image_tensor * 2.0 - 1.0
    return image_tensor.to(device=DEVICE, dtype=DTYPE)


def diy_img2img(
    pipe,
    init_image,
    prompt,
    negative_prompt="",
    strength=0.6,
    num_inference_steps=30,
    guidance_scale=7.5,
    seed=42,
    height=512,
    width=512,
):
    """
    Спрощена ручна реалізація Img2Img.
    """
    generator = torch.Generator(device=DEVICE).manual_seed(seed)

    # 1. PIL image → tensor → VAE latents
    image_tensor = prepare_pil_image_for_vae(init_image, height=height, width=width)

    with torch.no_grad():
        init_latents = pipe.vae.encode(image_tensor).latent_dist.mean
        init_latents = init_latents * pipe.vae.config.scaling_factor

    # 2. Prompt embeddings
    text_embeddings = get_prompt_embeds(
        pipe,
        prompt=prompt,
        negative_prompt=negative_prompt,
        num_images_per_prompt=1,
    ).to(dtype=DTYPE)

    # 3. Scheduler timesteps
    pipe.scheduler.set_timesteps(num_inference_steps, device=DEVICE)

    init_timestep = min(int(num_inference_steps * strength), num_inference_steps)
    t_start = max(num_inference_steps - init_timestep, 0)
    timesteps = pipe.scheduler.timesteps[t_start:]

    if len(timesteps) == 0:
        raise ValueError("strength занадто малий: немає denoising timesteps.")

    # 4. Додаємо шум до image latents
    noise = torch.randn(
        init_latents.shape,
        generator=generator,
        device=DEVICE,
        dtype=DTYPE,
    )

    latent_timestep = timesteps[:1]
    latents = pipe.scheduler.add_noise(init_latents, noise, latent_timestep)

    # 5. Denoising loop
    for t in timesteps:
        latent_model_input = torch.cat([latents] * 2)
        latent_model_input = pipe.scheduler.scale_model_input(latent_model_input, t)

        with torch.no_grad():
            noise_pred = pipe.unet(
                latent_model_input,
                t,
                encoder_hidden_states=text_embeddings,
            ).sample

        noise_pred_uncond, noise_pred_text = noise_pred.chunk(2)
        noise_pred = noise_pred_uncond + guidance_scale * (
            noise_pred_text - noise_pred_uncond
        )

        latents = pipe.scheduler.step(noise_pred, t, latents).prev_sample

    # 6. Latents → PIL image
    return decode_latents_to_pil(pipe, latents)[0]

In [ ]:
if RUN_IMG2IMG:
    #diy_img2img_prompt = "An oil painting of a person sitting on a park bench, warm light, high detail. The person sits facing the viewer."
    diy_img2img_prompt = (
       "A front-facing person sitting on a park bench, looking at the viewer, "
       "highly detailed face, natural skin, "
       "oil painting, warm light, high detail"
    )
    
    negative_prompt = (
       "back view, distorted face, from behind, turned away, rear view, no face, blurry, low quality,"
       "blurry face, deformed face, distorted face, bad eyes, bad nose, bad mouse, malformed eyes, poorly drawn face,"
       "extra fingers, low quality, blurry, ugly"
    )
    
    diy_img2img_image = diy_img2img(
        pipe=pipe,
        init_image=init_image,
        prompt=diy_img2img_prompt,
        negative_prompt= negative_prompt,
        strength=0.60,
        num_inference_steps=30,
        guidance_scale=7.5,
        seed=42,
        height=512,
        width=512,
    )

    show_images(
        [init_image, diy_img2img_image],
        titles=["Input image", "DIY Img2Img result"],
        cols=2,
        figsize=(10, 5)
    )

    save_images(diy_img2img_image, "diy_img2img")

## 18.2 Готовий `StableDiffusionImg2ImgPipeline`

Після ручного розбору використовуємо готовий pipeline.  
У практичній роботі саме цей варіант є основним, бо він коротший і стабільніший.

In [ ]:
if RUN_IMG2IMG:
    clear_gpu()

    img2img_pipe = StableDiffusionImg2ImgPipeline.from_pretrained(
        MODEL_ID,
        torch_dtype=DTYPE,
        use_safetensors=True,
    ).to(DEVICE)

    img2img_pipe.enable_attention_slicing()

    print_pipeline(img2img_pipe)
    try:
        img2img_pipe.enable_xformers_memory_efficient_attention()
    except Exception:
        pass

    print("Img2Img pipeline loaded:", type(img2img_pipe).__name__)

In [ ]:
if RUN_IMG2IMG:
   
    img2img_prompt = (
       "A front-facing person sitting on a park bench, looking at the viewer, "
       "highly detailed face, natural skin, "
       "oil painting, warm light, high detail"
    )
    
    img2img_negative_prompt = (
       "back view, distorted face, from behind, turned away, rear view, no face, blurry, low quality,"
       "blurry face, deformed face, distorted face, bad eyes, bad nose, bad mouse, malformed eyes, poorly drawn face,"
       "extra fingers,extra legs, low quality, blurry, ugly"
    )
    strengths = [0.25, 0.55, 0.85]
    img2img_results = []
    img2img_titles = []

    for strength in strengths:
        generator = torch.Generator(device=DEVICE).manual_seed(42)

        result = img2img_pipe(
            prompt=img2img_prompt,
            negative_prompt=img2img_negative_prompt,
            image=init_image,
            strength=strength,
            guidance_scale=7.5,
            num_inference_steps=30,
            generator=generator,
        ).images[0]

        img2img_results.append(result)
        img2img_titles.append(f"strength={strength}")

    show_images([init_image] + img2img_results, titles=["Input"] + img2img_titles, cols=4, figsize=(18, 5))
    save_images(img2img_results, "img2img_strength_experiment")

## Контрольні питання до Img2Img

1. Чим Img2Img відрізняється від Text-to-Image?
2. Що робить параметр `strength`?
3. При якому `strength` input image зберігається найкраще?
4. При якому `strength` prompt має найбільший вплив?

# 19. Inpainting pipeline

`Inpainting` потрібен, коли треба змінити тільки частину зображення.

Використовується:

- `image` — початкове зображення;
- `mask_image` — маска;
- `prompt` — що згенерувати у masked-області.

## Правило маски

```text
white area  → замінити
black area  → зберегти
```

У цьому notebook ми використовуємо `StableDiffusionInpaintPipeline`.

In [ ]:
if RUN_INPAINTING:
    clear_gpu()

    INPAINT_MODEL_ID = "stable-diffusion-v1-5/stable-diffusion-inpainting"

    inpaint_pipe = StableDiffusionInpaintPipeline.from_pretrained(
        INPAINT_MODEL_ID,
        torch_dtype=DTYPE
    ).to(DEVICE)

    inpaint_pipe.enable_attention_slicing()

    print_pipeline(inpaint_pipe)

    try:
        inpaint_pipe.enable_xformers_memory_efficient_attention()
    except Exception:
        pass

    print("Inpainting pipeline loaded:", type(inpaint_pipe).__name__)

In [ ]:
if RUN_INPAINTING:
    inpaint_prompt = "A small friendly robot, high resolution, sitting on a park bench"
    inpaint_negative_prompt = "blurry, low quality, distorted, bad anatomy"

    generator = torch.Generator(device=DEVICE).manual_seed(42)

    inpaint_result = inpaint_pipe(
        prompt=inpaint_prompt,
        negative_prompt=inpaint_negative_prompt,
        image=init_image,
        mask_image=mask_image,
        guidance_scale=8.0,
        num_inference_steps=30,
        generator=generator,
    ).images[0]

    show_images(
        [init_image, mask_image, inpaint_result],
        titles=["Input image", "Mask", "Inpainting result"],
        cols=3,
        figsize=(15, 5)
    )

    save_images(inpaint_result, "inpainting_result")

## Контрольні питання до Inpainting

1. Яка частина маски редагується: біла чи чорна?
2. Чому inpainting потребує і image, і mask?
3. Як зміниться результат, якщо маску зробити більшою?
4. Що буде, якщо prompt не відповідає формі masked-області?

# 20. Depth2Img pipeline

`Depth2Img` використовує depth map початкового зображення.  
Це допомагає зберігати просторову структуру, але змінювати стиль і зміст.

Цей блок опційний, бо завантажує окрему модель:

```text
 sd2-community/stable-diffusion-2-depth
```

Якщо GPU або disk space обмежені, залиште:

```python
RUN_DEPTH2IMG = False
```

In [ ]:
if RUN_DEPTH2IMG:
    clear_gpu()

    DEPTH_MODEL_ID = "sd2-community/stable-diffusion-2-depth"

    depth_pipe = StableDiffusionDepth2ImgPipeline.from_pretrained(
        DEPTH_MODEL_ID,
        torch_dtype=DTYPE       
    ).to(DEVICE)

    depth_pipe.enable_attention_slicing()

    print("Depth2Img pipeline loaded:", type(depth_pipe).__name__)

    print_pipeline(depth_pipe)
    
    depth_prompt = "An oil painting of a person sitting on a bench, cinematic lighting"
    generator = torch.Generator(device=DEVICE).manual_seed(42)

    depth_result = depth_pipe(
        prompt=depth_prompt,
        image=init_image,
        negative_prompt="blurry, low quality, distorted",
        strength=0.75,
        guidance_scale=7.5,
        num_inference_steps=30,
        generator=generator,
    ).images[0]

    show_images(
        [init_image, depth_result],
        titles=["Input image", "Depth2Img result"],
        cols=2,
        figsize=(10, 5)
    )

    save_images(depth_result, "depth2img_result")
else:
    print("Depth2Img пропущено. Щоб запустити, встановіть RUN_DEPTH2IMG = True у секції налаштувань.")

# 21. Порівняння pipeline

Заповніть таблицю у звіті.

| Pipeline | Стартова точка | Основні параметри | Що контролює результат |
|---|---|---|---|
| Text-to-Image |  |  |  |
| DIY sampling |  | |  |
| Img2Img | |  |  |
| Inpainting |  |  |  |
| Depth2Img |  |  |  |

## Завдання

1. Виберіть 3 отримані результати.
2. Опишіть, що збереглося від input image.
3. Опишіть, що змінилося під впливом prompt.
4. Вкажіть, який pipeline найкраще підходить для редагування частини зображення.

# 22. Рівні складності лабораторної

## Рівень “Задовільно”

Студент має:

- запустити notebook на GPU;
- виконати всі розділи і завдання під ними та відповісти на всі запитання в кінці розділів;
- пояснити параметри `prompt`, `negative_prompt`, `guidance_scale`, `num_inference_steps`;
- пояснити експеримент з 3 значеннями `guidance_scale` для Text-to-Image;
- пояснити експеримент з 3 значеннями `strength` для Image-to-Image;
- порівняти Text-to-Image, Image-to-Image та Inpainting;
- заповнити звіт з п.23

## Рівень “Добре”

Усе з рівня “Задовільно”, а також:

- пояснити DIY sampling loop;
- змінити scheduler і порівняти результат;
- використати власне безпечне input image і mask;
- провести системний експеримент: `steps × guidance_scale`;
- додати свої результати у звіт
  
## Рівень “Відмінно”

Усе з рівнів "Задовільно" та "Добре" 
та виконати завдання з ноутбуку Dreambooth



# 23. Шаблон звіту

Скопіюйте цей шаблон у свій звіт і заповніть.

---

## 1. Тема роботи

Stable Diffusion pipelines у Diffusers: Text-to-Image, Img2Img, Inpainting, Depth2Img.

## 2. Мета роботи

Коротко опишіть, що ви вивчали.

## 3. Середовище

| Параметр | Значення |
|---|---|
| Kaggle GPU |  |
| Python version |  |
| PyTorch version |  |
| Diffusers version |  |
| Model ID |  |
| Device |  |

## 4. Text-to-Image

Prompt:

```text

```

Negative prompt:

```text

```

Параметри:

| Parameter | Value |
|---|---|
| guidance_scale |  |
| num_inference_steps |  |
| seed |  |
| height × width |  |

Короткий аналіз результату:

```text

```

## 5. Pipeline components

Опишіть роль:

- VAE:
- Tokenizer:
- Text encoder:
- UNet:
- Scheduler:

## 6. VAE experiment

| Tensor | Shape |
|---|---|
| input image |  |
| raw latents |  |
| scaled latents |  |
| decoded image |  |

Поясніть `scaling_factor`:

```text

```

## 7. Img2Img

Порівняння `strength`:

| strength | Що збереглося | Що змінилося | Якість |
|---:|---|---|---|
| 0.25 |  |  |  |
| 0.55 |  |  |  |
| 0.85 |  |  |  |

## 8. Inpainting

Опишіть:

- яку область редагували;
- який prompt використали;
- чи добре зберігся фон;
- які артефакти виникли.

## 9. Порівняння pipeline

| Pipeline | Переваги | Недоліки | Коли використовувати |
|---|---|---|---|
| Text-to-Image |  |  |  |
| Img2Img |  |  |  |
| Inpainting |  |  |  |
| Depth2Img |  |  |  |

## 10. Етичний висновок

Опишіть, які ризики має генерація та редагування зображень:

```text

```

## 11. Загальний висновок

```text

```

# 24. Troubleshooting

## CUDA out of memory

Спробуйте:

```python
IMAGE_HEIGHT = 384
IMAGE_WIDTH = 384
```

або зменшити:

```python
num_inference_steps = 20
```

Також можна вимкнути важкі секції:

```python
RUN_INPAINTING = False
RUN_DEPTH2IMG = False
```

## Модель не завантажується

Перевірте:

```text
Kaggle Internet: On
HF_TOKEN: створений і доданий у Secrets
ліцензію моделі прийнято на Hugging Face
```

## Notebook зависає на pip install

Це може зайняти кілька хвилин. Якщо занадто довго:

1. Interrupt execution.
2. Restart session.
3. Запустіть cells по черзі.

## Результати не відтворюються

Перевірте, що використовується однаковий:

```python
seed
prompt
negative_prompt
guidance_scale
num_inference_steps
scheduler
model_id
```

# 25. Джерела

1. Hugging Face Diffusion Course, Unit 3, lesson 2:  
   https://huggingface.co/learn/diffusion-course/unit3/2

2. Diffusers documentation:  
   https://huggingface.co/docs/diffusers

3. Stable Diffusion pipelines in Diffusers:  
   https://huggingface.co/docs/diffusers/api/pipelines/stable_diffusion/overview

4. Hugging Face Hub authentication:  
   https://huggingface.co/docs/huggingface_hub